# Bin Buddy · WS1 · Notebook 01 — Data preparation and exploratory analysis

**Owner:** WS1 See (vision and sensing). **Goal:** one clean, leak-free, licence-tracked image
manifest that every model in notebook 02 is trained and tested on.

What this notebook does:
1. Downloads the three public datasets named in the proposal (TrashNet, RealWaste, TACO).
2. Adds the team's own Singapore phone photos (the local dataset).
3. Maps every source class onto the **10 material classes** of the shared contract.
4. Removes exact and near-duplicate images and splits **by group** (item / source photo), so
   the same object never appears in both training and test data.
5. Explores the data: class balance, image size, brightness and sharpness per source
   (the "domain gap"), and sample images.

Outputs (on Google Drive, `BinBuddy_Data/`): `prepared_v1.tar` (resized images),
`manifests/vision_manifest.csv`, `manifests/eda_summary.json`.

### Scope decision: the model predicts *material*, the questions establish *cleanliness*
The vision model answers **"what is this item made of?"** (paper, plastic, glass, metal, carton,
styrofoam, e-waste, textile, food waste, other). Whether a container is clean is asked through
the question bank (Q2 "Did you clean or rinse it?", Q3 "How long has it been sitting?").
Reason: none of the public datasets labels cleanliness, and ~100–200 local photos are far too
few to learn it reliably. Asking is more honest than guessing; the report states this deviation
from proposal objective O1.

## 0. Setup

Run on **Google Colab** (Runtime → Change runtime type → **T4 GPU** for notebooks 02–04; CPU is
enough for 01). The cell below mounts Google Drive, clones the team repository (branch
`ws1/vision-model`) and installs pinned packages, so every run uses the same code and versions.

`SMOKE=1` is only for the developer's local test run on tiny synthetic data; leave it unset on Colab.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

SMOKE = os.environ.get("BINBUDDY_SMOKE") == "1"  # local test on synthetic data only
REPO_URL = "https://github.com/DevelopingGod/Group-5-PRS-Practice-Module-2026.git"
BRANCH = "ws1/vision-model"
PINNED = ["timm==1.0.30", "grad-cam==1.5.7", "open-clip-torch==3.3.0", "imagehash==4.3.2"]

if SMOKE:
    REPO = Path(os.environ["BINBUDDY_REPO"])
    DATA = Path(os.environ["BINBUDDY_DATA_ROOT"])
    WORK = DATA / "_work"
else:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO = Path("/content/repo")
    DATA = Path("/content/drive/MyDrive/BinBuddy_Data")
    WORK = Path("/content/work")
    if not REPO.exists():
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)

# The contracts and helpers come straight from the repository (no copy-paste drift).
for p in (REPO / "contracts", REPO / "ml", REPO / "services" / "inference"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
WORK.mkdir(parents=True, exist_ok=True)
PREP = WORK / "prepared"  # resized images, extracted from DATA / "prepared_v1.tar"
MANIFEST = DATA / "manifests" / "vision_manifest.csv"
RESULTS = DATA / "results"
MODELS_DIR = DATA / "models"
for d in (MANIFEST.parent, RESULTS, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("repo:", REPO, "| data:", DATA, "| smoke:", SMOKE)

In [ ]:
import json
import shutil
import tarfile
import urllib.request
import zipfile
from collections import Counter

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageOps

from app.quality import check_quality
from binbuddy_contracts.enums import Material
from binbuddy_ml import public_data as pdm
from binbuddy_ml import taco as taco_dl

MAX_SIDE = 512  # store images at <= 512 px: enough for 224 px training crops, 10x less disk
LOCAL_DIR = DATA / "local_by_material"  # team photos, one sub-folder per material
MATERIALS = [m.value for m in Material]

## 1. Sources, licences and how their classes map to our materials

| Source | Images | Licence | What it contributes |
|---|---|---|---|
| TrashNet | 2,527 photos (the resized copy from the Hub; the repo also holds the same photos at full size) | MIT | Clean single items on a white board: easy, studio-like |
| RealWaste | 4,752 | CC BY-NC-SA 4.0 (non-commercial, academic use only) | Real, dirty landfill items incl. food and textiles |
| TACO | 1,500 photos, 4,784 boxes | CC BY 4.0 | Litter in real scenes; each box is cropped to one item |
| Local (team) | ~100–200 | Team-owned | Singapore packaging: cartons, bubble-tea cups, takeaway boxes |

The class mapping lives in `ml/binbuddy_ml/public_data.py` (unit-tested in CI). Two choices
worth explaining:
- **TACO exclusions**: cigarettes (667 tiny boxes would swamp *other general*), unlabelled
  litter (unknown material), broken glass (not a household item), foil blister packs
  (mixed material).
- **TACO box filter**: a box is kept only if its shorter side is ≥ 96 px and it covers ≥ 1% of
  the photo, so crops resemble a phone photo of one item rather than a speck of litter.

In [ ]:
cov = pdm.coverage(pdm.TRASHNET, pdm.REALWASTE, pdm.TACO)
pd.DataFrame(
    {"public source classes": [", ".join(cov[m]) or "NONE" for m in Material]}, index=MATERIALS
)

**Gap to note:** *e-waste* has only TACO's "Battery" boxes, and *beverage carton* and
*styrofoam* are thin. These classes depend on the local photos — so ask teammates to photograph
chargers, cables, light bulbs, small appliances, milk and drink cartons, and foam boxes.
Notebook 02 also tests zero-shot CLIP, which can recognise a class with no training images.

In [ ]:
def save_resized(img: Image.Image, out: Path) -> tuple[int, int]:
    """Upright, RGB, longest side <= MAX_SIDE, JPEG. Re-saving also drops EXIF (incl. GPS)."""
    if out.exists():
        with Image.open(out) as done:
            return done.size
    img = ImageOps.exif_transpose(img).convert("RGB")
    img.thumbnail((MAX_SIDE, MAX_SIDE))
    out.parent.mkdir(parents=True, exist_ok=True)
    img.save(out, "JPEG", quality=90)
    return img.size


def row(rel, source, source_class, material, group_id, licence, size):
    return {
        "path": str(rel).replace("\\", "/"),
        "source": source,
        "source_class": source_class,
        "material": material.value,
        "group_id": group_id,
        "licence": licence,
        "width": size[0],
        "height": size[1],
    }


def fake_image(seed: int) -> Image.Image:
    """Synthetic picture for SMOKE runs only."""
    rng = np.random.default_rng(seed)
    img = Image.new("RGB", (640, 480), tuple(int(v) for v in rng.integers(0, 255, 3)))
    d = ImageDraw.Draw(img)
    for _ in range(6):
        x, y = (int(v) for v in rng.integers(0, 500, 2))
        d.ellipse([x, y, x + 120, y + 90], fill=tuple(int(v) for v in rng.integers(0, 255, 3)))
    return img


def fake_rows(source, mapping, licence, per_class=6):
    rows = []
    for ci, name in enumerate(n for n, m in mapping.items() if m is not None):
        for i in range(per_class):
            rel = Path(source) / f"{ci:02d}-{i}.jpg"
            size = save_resized(fake_image(hash((source, ci, i)) % 2**32), PREP / rel)
            rows.append(row(rel, source, name, mapping[name], f"{source}-{ci}-{i}", licence, size))
    return rows

## 2. Download and prepare each source
### 2.1 TrashNet (Hugging Face Hub)

In [ ]:
def prepare_trashnet():
    if SMOKE:
        return fake_rows("trashnet", pdm.TRASHNET, "MIT")
    from huggingface_hub import hf_hub_download

    # The Hub repo holds the same 2,527 photos twice (original 3.6 GB, resized 43 MB). Only the
    # resized copy (512 x 384) is used: everything is stored at <= 512 px anyway.
    zpath = hf_hub_download("garythung/trashnet", "dataset-resized.zip", repo_type="dataset")
    raw = WORK / "trashnet_raw"
    if not raw.exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall(raw)
    dirs = {
        p.name: p
        for p in raw.rglob("*")
        if p.is_dir() and p.name in pdm.TRASHNET and "__MACOSX" not in p.parts
    }
    missing = set(pdm.TRASHNET) - set(dirs)
    assert not missing, f"TrashNet class folders not found: {missing}"
    rows = []
    for name, folder in sorted(dirs.items()):
        for f in sorted(
            x for x in folder.iterdir() if x.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ):
            rel = Path("trashnet") / name / f"tn-{f.stem}.jpg"
            with Image.open(f) as img:
                size = save_resized(img, PREP / rel)
            rows.append(row(rel, "trashnet", name, pdm.TRASHNET[name], f"tn-{f.stem}", "MIT", size))
    return rows


trashnet_rows = prepare_trashnet()
print(len(trashnet_rows), Counter(r["source_class"] for r in trashnet_rows))

### 2.2 RealWaste (UCI Machine Learning Repository)

In [ ]:
REALWASTE_URL = "https://archive.ics.uci.edu/static/public/908/realwaste.zip"


def prepare_realwaste():
    if SMOKE:
        return fake_rows("realwaste", pdm.REALWASTE, "CC BY-NC-SA 4.0")
    raw = WORK / "realwaste_raw"
    zpath = WORK / "realwaste.zip"
    if not zpath.exists():
        urllib.request.urlretrieve(REALWASTE_URL, zpath)
    if not raw.exists():
        with zipfile.ZipFile(zpath) as z:
            z.extractall(raw)
        for inner in list(raw.rglob("*.zip")):  # the archive may wrap a second zip
            with zipfile.ZipFile(inner) as z:
                z.extractall(inner.parent)
    dirs = {p.name: p for p in raw.rglob("*") if p.is_dir() and p.name in pdm.REALWASTE}
    missing = set(pdm.REALWASTE) - set(dirs)
    assert not missing, f"RealWaste class folders not found: {missing}"
    rows = []
    for name, folder in sorted(dirs.items()):
        files = sorted(f for f in folder.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
        for f in files:
            rel = Path("realwaste") / name.replace(" ", "_") / (f.stem + ".jpg")
            with Image.open(f) as img:
                size = save_resized(img, PREP / rel)
            gid = f"rw-{name}-{f.stem}"
            rows.append(
                row(rel, "realwaste", name, pdm.REALWASTE[name], gid, "CC BY-NC-SA 4.0", size)
            )
    return rows


realwaste_rows = prepare_realwaste()
print(len(realwaste_rows), Counter(r["source_class"] for r in realwaste_rows))

### 2.3 TACO (photos downloaded from Flickr by TACO's own script)
Some Flickr links may have died since 2019; missing photos are counted, not silently ignored.
Each kept box is cropped with 10% padding. **All crops from one photo share a group id**, so they
always land in the same split (otherwise the background would leak between train and test).

In [ ]:
def taco_fake():
    rows = []
    names = [n for n, m in pdm.TACO.items() if m is not None]
    for i in range(40):
        name = names[i % len(names)]
        rel = Path("taco") / f"{i:03d}.jpg"
        size = save_resized(fake_image(1000 + i), PREP / rel)
        rows.append(row(rel, "taco", name, pdm.TACO[name], f"taco-{i // 2}", "CC BY 4.0", size))
    return rows, Counter()


def prepare_taco():
    if SMOKE:
        return taco_fake()
    repo = WORK / "TACO"
    if not repo.exists():
        url = "https://github.com/pedropro/TACO.git"
        subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(repo)], check=True)
    ann_path = repo / "data" / "annotations.json"
    ann = json.loads(ann_path.read_text())
    # Our own downloader (binbuddy_ml.taco): TACO's download.py stops at the first dead Flickr
    # link. Each photo is tried at original size, then at 640 px; re-running skips finished ones.
    dl = taco_dl.download_all(ann["images"], repo / "data")
    print("TACO photos:", dict(dl))
    cats = {c["id"]: c["name"] for c in ann["categories"]}
    unknown = set(cats.values()) - set(pdm.TACO)
    assert not unknown, f"TACO categories missing from the mapping: {unknown}"
    images = {im["id"]: im for im in ann["images"]}
    by_image = {}
    for a in ann["annotations"]:
        by_image.setdefault(a["image_id"], []).append(a)
    stats, rows = Counter(), []
    for image_id, anns in by_image.items():
        meta = images[image_id]
        path = repo / "data" / meta["file_name"]
        if not path.exists():
            stats["photo missing (dead Flickr link)"] += len(anns)
            continue
        with Image.open(path) as raw:
            # Boxes are in original-photo pixels: match the orientation, scale to this file.
            img, scale = taco_dl.oriented(raw, meta["width"], meta["height"])
            img = img.convert("RGB")
            for k, a in enumerate(anns):
                name = cats[a["category_id"]]
                material = pdm.TACO[name]
                x, y, w, h = a["bbox"]
                if material is None:
                    stats[f"excluded: {name}"] += 1
                    continue
                if not pdm.taco_box_usable(w, h, meta["width"], meta["height"]):
                    stats["box too small"] += 1
                    continue
                if min(w, h) * scale < 64:  # only a small 640 px copy could be downloaded
                    stats["box too small in the 640 px copy"] += 1
                    continue
                rel = Path("taco") / f"taco-{image_id:05d}-{k:02d}.jpg"
                size = save_resized(taco_dl.crop_box(img, (x, y, w, h), scale), PREP / rel)
                gid = f"taco-{image_id}"
                rows.append(row(rel, "taco", name, material, gid, "CC BY 4.0", size))
                stats["kept"] += 1
    return rows, stats


taco_rows, taco_stats = prepare_taco()
print(len(taco_rows), "crops kept")
pd.Series(taco_stats).sort_values(ascending=False).head(15)

### 2.4 Local Singapore photos (the team's own)
Put every team photo into Google Drive as `BinBuddy_Data/local_by_material/<material>/`, where
`<material>` is one of: `paper, plastic, glass, metal, beverage_carton, styrofoam, e_waste,
textile, food_waste, other_general`. File names do not matter. Use JPEG or PNG
(iPhone HEIC: export as JPEG first). Rules: item only, no faces, no addresses, location
tagging off. Re-saving here strips all EXIF metadata, including GPS.

In [ ]:
def prepare_local():
    if SMOKE and not LOCAL_DIR.exists():
        for mi, m in enumerate(MATERIALS):
            for i in range(5):
                out = LOCAL_DIR / m / f"photo{i}.jpg"
                out.parent.mkdir(parents=True, exist_ok=True)
                fake_image(5000 + 10 * mi + i).save(out)
    if not LOCAL_DIR.exists():
        print("No local photos yet:", LOCAL_DIR)
        return [], []
    unknown = [p.name for p in LOCAL_DIR.iterdir() if p.is_dir() and p.name not in MATERIALS]
    assert not unknown, f"Folder names must be material names; fix these: {unknown}"
    rows, skipped = [], []
    for m in MATERIALS:
        folder = LOCAL_DIR / m
        if not folder.exists():
            continue
        for i, f in enumerate(sorted(folder.iterdir())):
            if f.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                skipped.append(f.name)
                continue
            rel = Path("local") / m / f"local-{m}-{i:04d}.jpg"
            with Image.open(f) as img:
                size = save_resized(img, PREP / rel)
            gid = f"local-{m}-{i:04d}"  # refined to real items in section 3
            rows.append(row(rel, "local", f.name, Material(m), gid, "team", size))
    return rows, skipped


local_rows, local_skipped = prepare_local()
print(
    len(local_rows),
    "local photos;",
    len(local_skipped),
    "skipped (not JPEG/PNG):",
    local_skipped[:10],
)
print(Counter(r["material"] for r in local_rows))

### 2.5 Extra web images (training only)
A few downloaded images fill classes the team photos lack (food waste, e-waste, styrofoam,
textile). Their licences are unknown, so they are used **only for training**: they never enter a
validation or test split, they are kept as their own source `web`, and they are left out of the
submitted dataset. Folder: `BinBuddy_Data/web_by_material/<material>/` (optional).

In [ ]:
WEB_DIR = DATA / "web_by_material"


def prepare_web():
    if SMOKE and not WEB_DIR.exists():
        for i, m in enumerate(["e_waste", "food_waste", "styrofoam", "textile"]):
            (WEB_DIR / m).mkdir(parents=True, exist_ok=True)
            fake_image(9000 + i).save(WEB_DIR / m / "web-0.jpg")
    if not WEB_DIR.exists():
        return []
    rows = []
    for m in MATERIALS:
        for f in sorted((WEB_DIR / m).glob("*.jpg")) if (WEB_DIR / m).exists() else []:
            rel = Path("web") / m / f.name
            with Image.open(f) as img:
                size = save_resized(img, PREP / rel)
            licence = "unknown (web; training only; not redistributed)"
            rows.append(row(rel, "web", f.name, Material(m), f"web-{f.stem}", licence, size))
    return rows


web_rows = prepare_web()
print(len(web_rows), "web images (training only):", Counter(r["material"] for r in web_rows))

## 3. Leakage control: duplicates and items photographed more than once
Two kinds of hidden duplicates would inflate test scores:
1. **The same image twice** (e.g. repeated rows in a Hub copy, or a TrashNet image re-used
   elsewhere). Detected with a 64-bit **perceptual hash** (pHash: DCT of a 32×32 greyscale
   thumbnail, one bit per low-frequency coefficient above the median). Two images whose hashes
   differ in ≤ 4 of 64 bits are treated as the same picture.
2. **The same local item from several angles** (teammates often take 2–3 shots). pHash misses
   this, so local photos are grouped by **DINOv2 embedding similarity** (cosine ≥ 0.80): two
   views of one carton sit close together in a self-supervised feature space.

Duplicates are merged into one group (union-find), and the split is assigned per group with a
deterministic hash (`split_for_group`), so nothing can straddle train and test.

In [ ]:
rows = trashnet_rows + realwaste_rows + taco_rows + local_rows + web_rows
df = pd.DataFrame(rows)
hashes = []
for p in df["path"]:
    with Image.open(PREP / p) as img:
        hashes.append(int(str(imagehash.phash(img)), 16))
df["phash"] = [f"{h:016x}" for h in hashes]

parent = list(range(len(df)))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(i, j):
    parent[find(i)] = find(j)


# Images already in one group (TACO crops of one photo) start united.
for _, idx in df.groupby("group_id").indices.items():
    for j in idx[1:]:
        union(idx[0], j)

H = np.array(hashes, dtype=np.uint64)
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
NEAR_DUP_BITS = 4
dup_pairs = Counter()
for start in range(0, len(H), 512):
    block = H[start : start + 512, None] ^ H[None, :]
    dist = POPCOUNT[block.view(np.uint8)].reshape(*block.shape, 8).sum(-1)
    for i, j in zip(*np.nonzero(dist <= NEAR_DUP_BITS), strict=True):
        a, b = start + i, j
        if a < b and df.at[a, "group_id"] != df.at[b, "group_id"]:
            dup_pairs[(df.at[a, "source"], df.at[b, "source"])] += 1
            union(a, b)
print("near-duplicate pairs by source:", dict(dup_pairs))

In [ ]:
import timm
import torch

local_idx = df.index[df["source"] == "local"].to_numpy()
LOCAL_SAME_ITEM_COS = 0.999 if SMOKE else 0.80  # synthetic SMOKE images all look alike
if len(local_idx) > 1:
    enc = timm.create_model(
        "vit_small_patch14_dinov2.lvd142m", pretrained=True, num_classes=0, img_size=224
    ).eval()
    cfg = timm.data.resolve_data_config({}, model=enc)
    cfg["input_size"] = (3, 224, 224)  # the pretrained config says 518; the model is built for 224
    tf = timm.data.create_transform(**cfg)
    with torch.no_grad():
        feats = []
        for i in local_idx:
            with Image.open(PREP / df.at[i, "path"]) as img:
                feats.append(enc(tf(img.convert("RGB"))[None])[0])
        e = torch.nn.functional.normalize(torch.stack(feats), dim=1)
        sim = (e @ e.T).numpy()
    # Complete linkage: photos join one item only if *every* pair in it is similar. (Single
    # linkage / union-find would chain A~B~C into one item even when A and C look different.)
    from sklearn.cluster import AgglomerativeClustering

    item = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="complete",
        distance_threshold=1 - LOCAL_SAME_ITEM_COS,
    ).fit_predict(np.clip(1 - sim, 0, 2))
    for k in np.unique(item):
        members = local_idx[item == k]
        for j in members[1:]:
            union(members[0], j)

df["group"] = [df.at[find(i), "group_id"] for i in range(len(df))]
multi = df[df["source"] == "local"].groupby("group").size()
print(
    "local photos:",
    len(local_idx),
    "-> items:",
    len(multi),
    "| items with >1 photo:",
    (multi > 1).sum(),
)
if len(multi) and multi.max() > 6:
    print(
        f"WARNING: one 'item' has {multi.max()} photos - probably different items merged; raise LOCAL_SAME_ITEM_COS"
    )

**Check by eye:** the grid below shows local groups with more than one photo. If two different
items were merged, raise `LOCAL_SAME_ITEM_COS` (e.g. 0.85) and re-run from section 3; if
obvious repeats were not merged, lower it.

In [ ]:
groups = [g for g, n in multi.items() if n > 1][:8]
if groups:
    fig, axes = plt.subplots(len(groups), 4, figsize=(8, 2 * len(groups)), squeeze=False)
    for r, g in enumerate(groups):
        paths = df.loc[df["group"] == g, "path"].tolist()[:4]
        for c in range(4):
            ax = axes[r][c]
            ax.axis("off")
            if c < len(paths):
                ax.imshow(Image.open(PREP / paths[c]))
                ax.set_title(g if c == 0 else "", fontsize=7)
    plt.tight_layout()
    plt.show()

In [ ]:
def assign_split(group: str, source: str) -> str:
    if source == "web":
        return "train"  # unknown licence and studio-like: never used to measure the model
    fractions = pdm.LOCAL_SPLIT if source == "local" else pdm.PUBLIC_SPLIT
    return pdm.split_for_group(group, fractions).value


def group_source(s):
    # a group touching web images stays in training; one touching team photos uses the local split
    for source in ("web", "local"):
        if source in set(s):
            return source
    return s.iloc[0]


first_source = df.groupby("group")["source"].agg(group_source)
df["split"] = [assign_split(g, first_source[g]) for g in df["group"]]
leak = df.groupby("group")["split"].nunique().max()
assert leak == 1, "a group straddles two splits"
pd.crosstab([df["source"]], df["split"], margins=True)

## 4. Exploratory data analysis
### 4.1 Class balance per source and split
Look for: classes that exist in only one source (the model may learn the *source*, e.g. the
white TrashNet background, instead of the material), and classes too thin in the local test set
to measure (macro-F1 on 3 images is noise).

In [ ]:
counts = pd.crosstab(df["material"], df["source"]).reindex(MATERIALS, fill_value=0)
counts.plot.barh(stacked=True, figsize=(8, 5), title="Images per material and source")
plt.xlabel("images")
plt.tight_layout()
plt.show()
counts.assign(total=counts.sum(axis=1))

In [ ]:
pd.crosstab(
    df.loc[df["source"] == "local", "material"], df.loc[df["source"] == "local", "split"]
).reindex(MATERIALS, fill_value=0)

### 4.2 Image sensing: brightness and sharpness per source (the domain gap)
These are the same two measurements the app's **image-quality checker** uses
(`services/inference/app/quality.py`): sharpness = variance of the Laplacian (high = sharp;
blur removes the high-frequency edges the Laplacian responds to) and brightness = mean grey level.
Different distributions per source are a *domain gap*: a model trained on bright, sharp
TrashNet photos may fail on dim kitchen photos. This is why the local dataset matters
(RealWaste's authors report 49.69% vs 89.19% accuracy for pristine vs real-world training data).

In [ ]:
sample = pd.concat([g.sample(min(len(g), 300), random_state=0) for _, g in df.groupby("source")])
q = []
for p in sample["path"]:
    with Image.open(PREP / p) as img:
        r = check_quality(img.convert("RGB"))
    q.append((r.blur_score, r.brightness, r.ok))
sample[["sharpness", "brightness", "quality_ok"]] = q
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
sample.boxplot("sharpness", by="source", ax=axes[0], showfliers=False)
axes[0].set_yscale("log")
sample.boxplot("brightness", by="source", ax=axes[1], showfliers=False)
plt.suptitle("")
plt.tight_layout()
plt.show()
sample.groupby("source")[["sharpness", "brightness", "quality_ok"]].median()

In [ ]:
df.groupby("source")[["width", "height"]].describe().round(0)

### 4.3 What the classes look like (one row per material)

In [ ]:
fig, axes = plt.subplots(len(MATERIALS), 6, figsize=(9, 1.6 * len(MATERIALS)))
for r, m in enumerate(MATERIALS):
    sub = df[df["material"] == m]
    picks = sub.sample(min(6, len(sub)), random_state=1) if len(sub) else sub
    for c in range(6):
        ax = axes[r][c]
        ax.axis("off")
        if c < len(picks):
            ax.imshow(Image.open(PREP / picks.iloc[c]["path"]))
            ax.set_title(picks.iloc[c]["source"], fontsize=6)
    axes[r][0].text(
        -0.1, 0.5, m, transform=axes[r][0].transAxes, ha="right", va="center", fontsize=8
    )
plt.tight_layout()
plt.show()

### 4.4 Findings (fill in after running — these go into the report)
- Class balance: …
- Classes with no or very little local data: …
- Domain gap (brightness / sharpness / background) between sources: …
- Duplicates removed and local items grouped: …
- Consequences for training (class weights, augmentation, which test set to trust): …

## 5. Save the manifest and the prepared images to Drive

In [ ]:
out = df.drop(columns=["group_id"])
out.to_csv(MANIFEST, index=False)
summary = {
    "images": len(out),
    "by_source": out["source"].value_counts().to_dict(),
    "by_split": out["split"].value_counts().to_dict(),
    "by_material": out["material"].value_counts().to_dict(),
    "local_items": int(out.loc[out["source"] == "local", "group"].nunique()),
    "taco_stats": dict(taco_stats),
    "near_duplicate_pairs": {f"{a}|{b}": n for (a, b), n in dup_pairs.items()},
}
(MANIFEST.parent / "eda_summary.json").write_text(json.dumps(summary, indent=2))
tar_path = DATA / "prepared_v1.tar"
with tarfile.open(tar_path, "w") as tf:
    tf.add(PREP, arcname=".")
print("saved", MANIFEST, "and", tar_path, f"({tar_path.stat().st_size / 1e6:.0f} MB)")
shutil.rmtree(WORK / "realwaste_raw", ignore_errors=True)